In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, iirnotch, welch
import glob

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils import resample, class_weight
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, VotingClassifier, BaggingClassifier, AdaBoostClassifier)
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# keras / tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, InputLayer
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical




In [6]:
# === CONCATENAZIONE DATASET ===
# Esc -> per uscire dalla modifica della cella
# M -> per trasformarla in commento
# Shift + Enter -> per formattarla (eventualmente)
# y -> per ritrasformarla in codice

# Lista dei nomi dei file CSV
file_list = [
    "PSD_dataset_Bergantin_Fulvio_42.csv",
    "PSD_dataset_Cambareri_Chiara_26.csv",
    "PSD_dataset_Capria_Francesco_26.csv",
    "PSD_dataset_Capria_Alessandro_19.csv",
    "PSD_dataset_Cicirelli_Franco_50.csv",
    "PSD_dataset_Coppolillo_Erica_25.csv",
    "PSD_dataset_Damore_Francesco_45.csv",
    "PSD_dataset_DeMaio_Annarita_34.csv",
    "PSD_dataset_Falcone_Alberto_40.csv",
    "PSD_dataset_Folino_Gianluigi_52.csv",
    "PSD_dataset_Guarascio_Massimo_42.csv",
    "PSD_dataset_Guerrieri_Antonio_42.csv",
    "PSD_dataset_Islam_Md_Babul.csv",
    "PSD_dataset_Khan_Irfanullah_33.csv",
    "PSD_dataset_Macrì_Davide_44.csv",
    "PSD_dataset_Mariani_Luca_33.csv",
    "PSD_dataset_Mastroianni_Carlo_53.csv",
    "PSD_dataset_Miceli_Massimo_50.csv",
    "PSD_dataset_Minici_Marco_29.csv",
    "PSD_dataset_Mungari_Simone_26.csv",
    "PSD_dataset_Pena_Hindred_27.csv",
    "PSD_dataset_Pisani_Sergio_Francesco_40.csv",
    "PSD_dataset_Qimeng_Li_37.csv",
    "PSD_dataset_Rizzo_Luigi_39.csv",
    "PSD_dataset_Scala_Francesco_33.csv",
    "PSD_dataset_Vinci_Andrea_40.csv",
    "PSD_dataset_Zicari_Paolo_51.csv"
]

# Leggi e concatena tutti i file
df_list = [pd.read_csv(file) for file in file_list]
psd_completo = pd.concat(df_list, ignore_index=True)
psd_completo.to_csv("PSD_dataset_27.csv", index=False)



In [2]:
# === CARICAMENTO DEL DATASET DAL CSV ===

psd_df = pd.read_csv("PSD_dataset_Capria_Francesco_26.csv")
#psd_df_15 = pd.read_csv("PSD_dataset_15.csv")
#psd_df_20 = pd.read_csv("PSD_dataset_20.csv")
#psd_df_c = pd.read_csv("PSD_dataset_completo.csv")

#print(psd_df_15.shape)
#print(psd_df_20.shape)
#print(psd_df_c.shape)



In [3]:
# === SPLIT DATASET ===

# === SPLIT ORIGINALE ===
X = psd_df.drop("Event Id", axis=1).astype(float)
y = psd_df["Event Id"]  # Nessun fillna necessario, lo hai già fatto



# Normalizzazione globale 
#X_scaled = preprocessing.StandardScaler().fit_transform(X)

# Split stratificato
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.8, shuffle=True, random_state=42, stratify=y    #se normalizzo tra zero e uno uso x_scaled_minmax, altrimenti x_scaled
)

# Normalizzazione tra 0 e 1
scaler = MinMaxScaler()
X_scaled_minmax = scaler.fit_transform(X_train)
X_scaled_test = scaler.transform(X_test)

# === COSTRUZIONE DATAFRAME PER IL TRAINING SET ===
train_df = pd.DataFrame(X_scaled_minmax)
train_df["Event Id"] = y_train.values


### ---UNDERSAMPLING CLASSE PIù FREQUENTE---
# Suddivisione per classi
df_0 = train_df[train_df["Event Id"] == 0.0]
df_others = train_df[train_df["Event Id"] != 0.0]

# Sottocampionamento classe 0.0
n_samples = int(df_others["Event Id"].value_counts().mean())
df_0_downsampled = resample(df_0, replace=False, n_samples=n_samples, random_state=42)

# Ricombinazione e shuffle
df_train_balanced_US = pd.concat([df_0_downsampled, df_others])
df_train_balanced_US = df_train_balanced_US.sample(frac=1, random_state=42)

# Recupero X e y bilanciati
X_train_balanced_US = df_train_balanced_US.drop("Event Id", axis=1).values
y_train_balanced_US = df_train_balanced_US["Event Id"].values


### ---OVERSAPLING CLASSI MINORITARIE---

# Trova la classe più numerosa
class_counts = train_df["Event Id"].value_counts()
max_samples = class_counts.max()

# Oversampling di tutte le classi (tranne quella già con più esempi)
balanced_dfs = []
for cls, count in class_counts.items():
    df_cls = train_df[train_df["Event Id"] == cls]
    if count < max_samples:
        df_upsampled = resample(df_cls, replace=True, n_samples=max_samples, random_state=42)
    else:
        df_upsampled = df_cls
    balanced_dfs.append(df_upsampled)

# Ricombinazione e shuffle
df_train_balanced_OS = pd.concat(balanced_dfs)
df_train_balanced_OS = df_train_balanced_OS.sample(frac=1, random_state=42)

# Recupero X e y bilanciati
X_train_balanced_OS = df_train_balanced_OS.drop("Event Id", axis=1).values
y_train_balanced_OS = df_train_balanced_OS["Event Id"].values



In [4]:
# === CLASSIFICAZIONE RANDOM FOREST ===

rcf=RandomForestClassifier(n_estimators=50, random_state=42, class_weight='balanced') 
rcf.fit(X_train_balanced_OS,y_train_balanced_OS)
y_hat_rf = rcf.predict(X_scaled_test)
y_actu_rf = pd.Series(y_test, name = 'Actual')
df_confusion = pd.crosstab(y_actu_rf, y_hat_rf)

# === VALUTAZIONE ===
acc = accuracy_score(y_test, y_hat_rf, normalize=True)
print('Model Accuracy: %.2f'%acc)
print(df_confusion)
print(classification_report(y_test,y_hat_rf))

Model Accuracy: 0.87
col_0    0.0      33024.0  33025.0  33026.0  33027.0
Actual                                              
0.0          123        0        1        0        4
33024.0       11       34        0        0        0
33025.0        7        0       38        0        0
33026.0       10        0        0       35        0
33027.0        6        0        0        1       38
              precision    recall  f1-score   support

         0.0       0.78      0.96      0.86       128
     33024.0       1.00      0.76      0.86        45
     33025.0       0.97      0.84      0.90        45
     33026.0       0.97      0.78      0.86        45
     33027.0       0.90      0.84      0.87        45

    accuracy                           0.87       308
   macro avg       0.93      0.84      0.87       308
weighted avg       0.89      0.87      0.87       308



In [6]:
# === CLASSIFICAZIONE CON SVM RBF ===
svm_rbf = SVC(kernel='rbf', class_weight='balanced')  # con bilanciamneto classi
#svm_rbf = SVC(kernel='rbf')  # senza bilanciamento classi
svm_rbf.fit(X_train_balanced_OS, y_train_balanced_OS)
y_hat_svm = svm_rbf.predict(X_scaled_test)
y_actu_svm = pd.Series(y_test, name = 'Actual')
df_confusion = pd.crosstab(y_actu_svm, y_hat_svm)

# === VALUTAZIONE ===
acc = accuracy_score(y_test, y_hat_svm, normalize=True)
print('Model Accuracy: %.2f'%acc)
print(df_confusion)
print(classification_report(y_test,y_hat_svm))


Model Accuracy: 0.92
col_0    0.0      33024.0  33025.0  33026.0  33027.0
Actual                                              
0.0          114        2        3        4        5
33024.0        2       43        0        0        0
33025.0        3        0       41        0        1
33026.0        3        0        0       41        1
33027.0        1        0        0        0       44
              precision    recall  f1-score   support

         0.0       0.93      0.89      0.91       128
     33024.0       0.96      0.96      0.96        45
     33025.0       0.93      0.91      0.92        45
     33026.0       0.91      0.91      0.91        45
     33027.0       0.86      0.98      0.92        45

    accuracy                           0.92       308
   macro avg       0.92      0.93      0.92       308
weighted avg       0.92      0.92      0.92       308



In [ ]:
### --- RETE NEURALE MLP (Multy Layer Perceptron)--- ###

# === CALLBACKS ===
checkpoint = ModelCheckpoint(
    filepath="best_model.keras",            # File dove salvare i pesi migliori
    monitor='val_loss',              # Valutazione in base alla accuratezza di validazione
    save_best_only=True,
    mode='min',
    verbose=0
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',                  # Se la val_loss non migliora...
    factor=0.5,                          # ...dimezza il learning rate
    patience=5,                          # dopo 5 epoche senza miglioramento
    min_lr=1e-6,                         # learning rate minimo
    verbose=0
)

callbacks_list = [checkpoint, reduce_lr]

# === PREPARAZIONE ONE-HOT ENCODING ===
#
classes = np.sort(np.unique(np.concatenate([y_train_balanced_OS, y_test])))
to_index = {c: i for i, c in enumerate(classes)}

y_train_cat = to_categorical([to_index[c] for c in y_train_balanced_OS],
                             num_classes=len(classes))
y_test_cat = to_categorical([to_index[c] for c in y_test],
                            num_classes=len(classes))


# === COSTRUZIONE MODELLO MLP ===

# Sequential Model
model = Sequential()

# Input Layer
model.add(InputLayer(shape=(X_train_balanced_OS.shape[1],)))

# L1 (hidden)
model.add(Dense(256, activation='relu'))   #tanh

# L2 (hidden)
model.add(Dense(128, activation='relu'))

# L3 (hidden)
model.add(Dense(64, activation='relu'))

# Lp (output)
model.add(Dense(len(classes), activation='softmax'))

# Inizializzazione del Modello
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# === ADDDESTRAMENTO ===
#es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

model.fit(X_train_balanced_OS, y_train_cat, validation_split=0.2, epochs=200, batch_size=32, callbacks=[checkpoint, reduce_lr], verbose=0)


# === PREDIZIONE ===
y_hat_nn = model.predict(X_scaled_test)
y_hat_nn_labels = classes[np.argmax(y_hat_nn, axis=1)]
y_actu_nn = pd.Series(y_test, name='Actual')
df_confusion_nn = pd.crosstab(y_actu_nn, y_hat_nn_labels)

# === VALUTAZIONE ===
acc_nn = accuracy_score(y_test, y_hat_nn_labels)
print("Model Accuracy (Neural Network): %.2f" % acc_nn)
print(df_confusion_nn)
print(classification_report(y_test, y_hat_nn_labels))


In [ ]:
'''# === CLASSIFICAZIONE CON KNN ===
knn = KNeighborsClassifier(n_neighbors=3, metric='manhattan') 
knn.fit(X_train_balanced_OS, y_train_balanced_OS)
y_hat_knn = knn.predict(X_scaled_test)
y_actu_knn = pd.Series(y_test, name = 'Actual')
df_confusion = pd.crosstab(y_actu_knn, y_hat_knn)
#print(y_pred)

# === VALUTAZIONE ===
acc = accuracy_score(y_test, y_hat_knn, normalize=True)
print('Model Accuracy: %.2f'%acc)
print(df_confusion)
print(classification_report(y_test, y_hat_knn))
'''


In [ ]:
'''# === CLASSIFICAZIONE CON LDA ===
lda = LinearDiscriminantAnalysis()
lda.fit(X_train_balanced_OS, y_train_balanced_OS)

y_hat_lda = lda.predict(X_scaled_test)
y_actu_lda = pd.Series(y_test, name='Actual')
df_confusion_lda = pd.crosstab(y_actu_lda, y_hat_lda)

# === VALUTAZIONE ===
acc_lda = accuracy_score(y_test, y_hat_lda)   # normalize=True è di default
print('Model Accuracy (LDA): %.2f' % acc_lda)
print(df_confusion_lda)
print(classification_report(y_test, y_hat_lda))'''


In [ ]:
'''###--- VOTING_CLASSIFIER---###
# Idea: combina più modelli diversi (es. Random Forest, SVM, LDA...), che poi andranno a votare per la classe finale.
# Hard Voting -> ogni modello fa una predizione, si prende la classe più votata.
# Soft Voting -> ogni modello restituisce la probabilità per ogni classe, e si fa la media delle probabilità.


# Definizione dei classificatori
log_clf = LogisticRegression(max_iter=1000)
rf_clf = RandomForestClassifier(n_estimators=200, class_weight='balanced')
svm_clf = SVC(kernel='rbf', probability=True, class_weight='balanced')
lda_clf = LinearDiscriminantAnalysis()

# Voting ensemble
voting_clf = VotingClassifier(
    estimators=[
        ('lr', log_clf),
        ('rf', rf_clf),
        ('svm', svm_clf),
        ('lda', lda_clf)
    ],
    voting='soft'  # usa 'hard' se vuoi la maggioranza delle etichette
)

# Addestramento
voting_clf.fit(X_train_balanced_OS, y_train_balanced_OS)

# Predizione
y_hat_vote = voting_clf.predict(X_scaled_test)
y_actu_vote = pd.Series(y_test, name='Actual')
df_confusion_vote = pd.crosstab(y_actu_vote, y_hat_vote)

# Valutazione
acc_vote = accuracy_score(y_test, y_hat_vote)
print("Model Accuracy (Voting Ensemble): %.2f" % acc_vote)
print(df_confusion_vote)
print(classification_report(y_test, y_hat_vote))'''


In [ ]:
'''###--- BAGGING (Bootstrapt Aggregating) ---###
# Idea: si utilizza lo stesso modello (Decision Tree), addestrandolo però su sottoinsiemi diversi dei dati.
# Ogni modello vede una porzione diversa del dataset.
# Alla fine poi le predizioni vengono aggregate (tramite voting)

bag_clf = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100, random_state=42)

# Addestramento
bag_clf.fit(X_train_balanced_OS, y_train_balanced_OS)

# Predizione
y_hat_bag = bag_clf.predict(X_scaled_test)
y_actu_bag = pd.Series(y_test, name='Actual')
df_confusion_bag = pd.crosstab(y_actu_bag, y_hat_bag)

# Valutazione
acc_bag = accuracy_score(y_test, y_hat_bag)
print("Model Accuracy (Bagging): %.2f" % acc_bag)
print(df_confusion_bag)
print(classification_report(y_test, y_hat_bag))'''


In [ ]:
'''### --- ADABOOST --- ###
# Idea: i modelli sono addestrati in sequenza, e ogni nuovo modello cerca di correggere di correggere gli errori del precedente
# i campioni più complessi (quelli classificati male) ricevono un peso maggiore

ada_clf = AdaBoostClassifier(n_estimators=50, random_state=42, learning_rate = 0.5, estimator = DecisionTreeClassifier(max_depth = 7))

# Addestramento
ada_clf.fit(X_train_balanced_OS, y_train_balanced_OS)

# Predizione
y_hat_ada = ada_clf.predict(X_scaled_test)
y_actu_ada = pd.Series(y_test, name='Actual')
df_confusion_ada = pd.crosstab(y_actu_ada, y_hat_ada)

# Valutazione
acc_ada = accuracy_score(y_test, y_hat_ada)
print("Model Accuracy (AdaBoost): %.2f" % acc_ada)
print(df_confusion_ada)
print(classification_report(y_test, y_hat_ada))'''
